# MLX例子 多层感知机

In [1]:
import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim

import numpy as np

继承nn.Module，nn为神经网络

### `__init__` 公式说明

记：

- $d_{\text{in}} = \texttt{input\_dim}$
- $d_h = \texttt{hidden\_dim}$
- $d_{\text{out}} = \texttt{output\_dim}$
- $L = \texttt{num\_layers}$（隐藏层个数）

层宽序列：

$$
\mathbf{s} = [d_{\text{in}},\; \underbrace{d_h,\; d_h,\; \ldots,\; d_h}_{L\text{ 个}},\; d_{\text{out}}]
$$

对应：

```python
layer_sizes = [input_dim] + [hidden_dim] * num_layers + [output_dim]
```

由此得到 $L+1$ 个线性层；第 $i$ 层（$i=1,\ldots,L+1$）为：

$$
f_i(\mathbf{z}) = W^{(i)}\mathbf{z} + \mathbf{b}^{(i)}
$$

其中

$$
W^{(i)} \in \mathbb{R}^{s_i \times s_{i-1}},\quad
\mathbf{b}^{(i)} \in \mathbb{R}^{s_i},\quad
s_j = \mathbf{s}[j]
$$

即：

- $W^{(1)} \in \mathbb{R}^{d_h \times d_{\text{in}}}$
- $W^{(2)},\ldots,W^{(L)} \in \mathbb{R}^{d_h \times d_h}$（当 $L \ge 2$）
- $W^{(L+1)} \in \mathbb{R}^{d_{\text{out}} \times d_h}$

前向对照（前 $L$ 层接 ReLU，最后一层无激活）：

$$
\begin{aligned}
\mathbf{h}^{(0)} &= \mathbf{x} \\
\mathbf{h}^{(i)} &= \mathrm{ReLU}\!\big(W^{(i)}\mathbf{h}^{(i-1)} + \mathbf{b}^{(i)}\big),\quad i=1,\ldots,L \\
\hat{\mathbf{y}} &= W^{(L+1)}\mathbf{h}^{(L)} + \mathbf{b}^{(L+1)}
\end{aligned}
$$

示例：`num_layers=2`, `hidden_dim=32`, MNIST `input_dim=784`, `output_dim=10` 时

$$
\mathbf{s} = [784, 32, 32, 10]
$$

即 $784\to 32$（ReLU）→ $32\to 32$（ReLU）→ $32\to 10$（无激活）。

In [3]:
class MLP(nn.Module):
    # 定义初始化函数
    def __init__(
        self, num_layers: int, input_dim: int, hidden_dim: int, output_dim: int
    ):
        super().__init__()
        layer_sizes = [input_dim] + [hidden_dim] * num_layers + [output_dim]
        # 定义线性层 这里需要画图理解
        self.layers = [
            nn.Linear(idim, odim)
            for idim, odim in zip(layer_sizes[:-1], layer_sizes[1:])
        ]

    # 定义前向传播函数
    def __call__(self, x):
        for l in self.layers[:-1]:
            x = mx.maximum(l(x), 0.0)
        return self.layers[-1](x)

In [4]:
def loss_fn(model, X, y):
    return mx.mean(nn.losses.cross_entropy(model(X), y))

In [5]:
def eval_fn(model, X, y):
    return mx.mean(mx.argmax(model(X), axis=1) == y)

In [9]:
# 层数
num_layers = 2
# 隐藏维度
hidden_dim = 32
# 分类数
num_classes = 10
# 批量大小
batch_size = 256
# 迭代次数
num_epochs = 10
# 训练速度
learning_rate = 1e-1

# Load the data (local mnist.py from mlx-examples, not the PyPI "mnist" package)
import sys
from pathlib import Path

_nb_dir = next(
    (
        p
        for p in (Path.cwd(), Path.cwd() / "notebooks" / "runtime")
        if (p / "mnist.py").exists()
    ),
    Path.cwd(),
)
sys.path.insert(0, str(_nb_dir))

import importlib
import mnist

importlib.reload(mnist)
print(mnist.__file__)
train_images, train_labels, test_images, test_labels = map(
    mx.array, mnist.mnist()
)

AttributeError: module 'mnist' has no attribute 'mnist'

In [12]:
def batch_iterate(batch_size, X, y):
    perm = mx.array(np.random.permutation(y.size))
    for s in range(0, y.size, batch_size):
        ids = perm[s : s + batch_size]
        yield X[ids], y[ids]

In [13]:
# Load the model
model = MLP(num_layers, train_images.shape[-1], hidden_dim, num_classes)
mx.eval(model.parameters())

# Get a function which gives the loss and gradient of the
# loss with respect to the model's trainable parameters
loss_and_grad_fn = nn.value_and_grad(model, loss_fn)

# Instantiate the optimizer
optimizer = optim.SGD(learning_rate=learning_rate)

for e in range(num_epochs):
    for X, y in batch_iterate(batch_size, train_images, train_labels):
        loss, grads = loss_and_grad_fn(model, X, y)

        # Update the optimizer state and model parameters
        # in a single call
        optimizer.update(model, grads)

        # Force a graph evaluation
        mx.eval(model.parameters(), optimizer.state)

    accuracy = eval_fn(model, test_images, test_labels)
    print(f"Epoch {e}: Test accuracy {accuracy.item():.3f}")

NameError: name 'train_images' is not defined